In [1]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage, SystemMessage

model = init_chat_model("deepseek-chat")

In [2]:
system_msg = SystemMessage("You are a helpful assistant.")
human_msg = HumanMessage("Hello, how are you?")

# Use with chat models
messages = [system_msg, human_msg]
response = model.invoke(messages)  # Returns AIMessage

In [3]:
response.pretty_print()

================================== Ai Message ==================================

Hello! I'm doing great, thank you for asking! 😊 How are you today? Is there anything I can help you with?


#### AIMessage

当模型调用工具时，它们会被包含在 AIMessage 中：

In [4]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."

model_with_tools = model.bind_tools([get_weather])

In [5]:
messages = [HumanMessage("What's the weather in Boston?")]
ai_msg = model_with_tools.invoke(messages)

In [7]:
ai_msg.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Boston'},
  'id': 'call_00_edc6qDWmAqys8dFdYOh1gwy1',
  'type': 'tool_call'}]

Token 使用情况

In [8]:
ai_msg.usage_metadata

{'input_tokens': 305,
 'output_tokens': 53,
 'total_tokens': 358,
 'input_token_details': {'cache_read': 256},
 'output_token_details': {}}

#### ToolMessage

In [10]:
from langchain.messages import ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = model.invoke(messages)  # Model processes the result

In [11]:
response.pretty_print()

================================== Ai Message ==================================

It's currently **sunny and 72°F** in San Francisco.
